In [2]:
import os, sys

os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot'
os.environ['PATH'] = os.environ['JAVA_HOME'] + r'\bin;' + os.environ['PATH']
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = '127.0.0.1'
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark import SparkContext, SparkConf
tmp = 'C:/spark-tmp'
os.makedirs(tmp, exist_ok=True)

conf = (SparkConf()
    .setAppName('CSL7110_Assignment4')
    .setMaster('local[1]')
    .set('spark.local.dir', tmp)
    .set('spark.driver.host', '127.0.0.1')
    .set('spark.driver.bindAddress', '127.0.0.1')
    .set('spark.network.timeout', '300s')
    .set('spark.executor.heartbeatInterval', '100s')
    .set('spark.python.worker.reuse', 'false')
    .set('spark.driver.extraJavaOptions',
         '-Djava.net.preferIPv4Stack=true '
         f'-Djava.io.tmpdir={tmp}')
    .set('spark.executor.extraJavaOptions',
         '-Djava.net.preferIPv4Stack=true '
         f'-Djava.io.tmpdir={tmp}'))

sc = SparkContext(conf=conf)
sc.setLogLevel('ERROR')
print("SC started:", sc.version)

SC started: 3.5.1


 ### Load graph and build adjacency RDD

In [3]:
def load_graph(sc, filepath):
    raw = sc.textFile(filepath)

    edges_rdd = (
        raw
        .map(lambda line: line.strip().split())
        .filter(lambda parts: len(parts) >= 2)
        .map(lambda parts: (int(parts[0]), int(parts[1]))) 
    )
    edges_rdd.cache()

    src_nodes = edges_rdd.map(lambda e: e[0]).distinct()
    dst_nodes = edges_rdd.map(lambda e: e[1]).distinct()
    all_nodes = src_nodes.union(dst_nodes).distinct().collect()
    n = len(all_nodes)

    return edges_rdd, all_nodes, n

print('load_graph defined successfully')

load_graph defined successfully


### Build the transition matrix as an RDD of (dst, contribution) pairs

In [4]:
def build_adjacency(edges_rdd):
    """
    Build adjacency representation: for each source node,
    compute out-degree and list of neighbors.

    Returns
    -------
    adj_rdd : RDD of (src, (out_degree, [dst1, dst2, ...]))
    """
    # Group edges by source: src -> [dst1, dst2, ...]
    neighbors = edges_rdd.groupByKey().mapValues(list)

    # Compute out-degree and attach to the neighbor list
    adj_rdd = neighbors.mapValues(lambda dsts: (len(dsts), dsts))
    adj_rdd.cache()
    return adj_rdd

print('build_adjacency defined successfully!')

build_adjacency defined successfully!


### Iterative PageRank

In [5]:
def pagerank(sc, edges_rdd, all_nodes, n, beta=0.8, num_iterations=40):
    teleport = (1.0 - beta) / n  
    adj_rdd = build_adjacency(edges_rdd).cache()
    ranks = sc.parallelize([(node, 1.0 / n) for node in all_nodes]).cache()
    
    for iteration in range(num_iterations):
        contributions = (
            adj_rdd
            .join(ranks)    
            .flatMap(lambda x: [
                (dst, x[1][1] / x[1][0][0])     
                for dst in x[1][0][1]
            ])
        )
        summed = contributions.reduceByKey(lambda a, b: a + b)
        all_nodes_rdd = sc.parallelize([(node, 0.0) for node in all_nodes])
        summed = (
            summed
            .union(all_nodes_rdd)
            .reduceByKey(lambda a, b: a + b)  
        )
        new_ranks = summed.mapValues(lambda contrib: teleport + beta * contrib).cache()
        
        ranks.unpersist()
        ranks = new_ranks
        
        # Force materialization every 10 iterations to break lineage
        if (iteration + 1) % 10 == 0:
            # Collect and re-parallelize
            ranks_local = ranks.collect()
            ranks.unpersist()
            ranks = sc.parallelize(ranks_local).cache()
            print(f'  Completed iteration {iteration + 1}/{num_iterations}')
    
    rank_dict = dict(ranks.collect())
    return rank_dict

### Run PageRank on small.txt

In [6]:

SMALL_GRAPH = 'small.txt'
WHOLE_GRAPH  = 'whole.txt'

BETA           = 0.8
NUM_ITERATIONS = 40

print('Running PageRank on small.txt...')
try:
    edges_small, nodes_small, n_small = load_graph(sc, SMALL_GRAPH)
    print(f'  Nodes: {n_small},  Edges: {edges_small.count()}')

    ranks_small = pagerank(sc, edges_small, nodes_small, n_small,
                           beta=BETA, num_iterations=NUM_ITERATIONS)

    sorted_small = sorted(ranks_small.items(), key=lambda x: -x[1])
    print(f'\nTop node score (small.txt): {sorted_small[0][1]:.6f}')

    print('\nTop 5 nodes (small graph):')
    for node, score in sorted_small[:5]:
        print(f'  Node {node:5d}  score = {score:.6f}')

    print('\nBottom 5 nodes (small graph):')
    for node, score in sorted_small[-5:]:
        print(f'  Node {node:5d}  score = {score:.6f}')

except Exception as e:
    print(f'Error loading small.txt: {e}')

Running PageRank on small.txt...
  Nodes: 100,  Edges: 1024
  Completed iteration 10/40
  Completed iteration 20/40
  Completed iteration 30/40
  Completed iteration 40/40

Top node score (small.txt): 0.037869

Top 5 nodes (small graph):
  Node    53  score = 0.037869
  Node    14  score = 0.035867
  Node     1  score = 0.035141
  Node    40  score = 0.033831
  Node    27  score = 0.033130

Bottom 5 nodes (small graph):
  Node    89  score = 0.003840
  Node    37  score = 0.003714
  Node    81  score = 0.003580
  Node    59  score = 0.003444
  Node    85  score = 0.003235


### Run PageRank on whole.txt

In [5]:
import time
WHOLE_GRAPH  = 'whole.txt'

BETA           = 0.8
NUM_ITERATIONS = 40
print('Running PageRank on whole.txt (n=1000, m=8192)...')
try:
    edges_whole, nodes_whole, n_whole = load_graph(sc, WHOLE_GRAPH)
    print(f'  Nodes: {n_whole},  Edges: {edges_whole.count()}')

    start = time.time()
    ranks_whole = pagerank(sc, edges_whole, nodes_whole, n_whole,
                           beta=BETA, num_iterations=NUM_ITERATIONS)
    elapsed = time.time() - start
    print(f'PageRank on whole.txt completed in {elapsed:.2f} seconds')

    sorted_whole = sorted(ranks_whole.items(), key=lambda x: -x[1])

    print('\nTop 5 nodes (whole graph):')
    for node, score in sorted_whole[:5]:
        print(f'  Node {node:5d}  score = {score:.6f}')

    print('\nBottom 5 nodes (whole graph):')
    for node, score in sorted_whole[-5:]:
        print(f'  Node {node:5d}  score = {score:.6f}')

except Exception as e:
    print(f'Error loading whole.txt: {e}')

Running PageRank on whole.txt (n=1000, m=8192)...
  Nodes: 1000,  Edges: 8192
  Completed iteration 10/40
  Completed iteration 20/40
  Completed iteration 30/40
  Completed iteration 40/40
PageRank on whole.txt completed in 2711.99 seconds

Top 5 nodes (whole graph):
  Node   263  score = 0.002017
  Node   537  score = 0.001944
  Node   965  score = 0.001894
  Node   243  score = 0.001850
  Node   187  score = 0.001830

Bottom 5 nodes (whole graph):
  Node   408  score = 0.000388
  Node    62  score = 0.000361
  Node   424  score = 0.000355
  Node    93  score = 0.000351
  Node   558  score = 0.000330


In [ ]:
sc.stop()